# Calendar Dimension Build

Creates the standard calendar table `adb_healthy.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31, including year, quarter, month, week, day and flag attributes.

In [ ]:
CREATE OR REPLACE TABLE adb_healthy.silver.calendar
COMMENT 'Standard calendar dimension, one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)                 AS date_key,
  d                                                       AS date,
  -- Year
  year(d)                                                 AS year,
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END               AS half_year,
  concat('H', CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END)  AS half_year_name,
  -- Quarter
  quarter(d)                                              AS quarter,
  concat('Q', quarter(d))                                 AS quarter_name,
  concat(year(d), '-Q', quarter(d))                       AS year_quarter,
  -- Month
  month(d)                                                AS month,
  date_format(d, 'MMMM')                                  AS month_name,
  date_format(d, 'MMM')                                   AS month_short_name,
  date_format(d, 'yyyy-MM')                               AS year_month,
  CAST(date_format(d, 'yyyyMM') AS INT)                   AS year_month_key,
  -- Week (ISO 8601, Monday start)
  weekofyear(d)                                           AS week_of_year,
  date_part('YEAROFWEEK', d)                              AS iso_week_year,
  concat(date_part('YEAROFWEEK', d), '-W', lpad(CAST(weekofyear(d) AS STRING), 2, '0')) AS iso_year_week,
  CAST(ceil(day(d) / 7.0) AS INT)                         AS week_of_month,
  -- Day
  day(d)                                                  AS day_of_month,
  dayofyear(d)                                            AS day_of_year,
  CAST(datediff(d, CAST(date_trunc('QUARTER', d) AS DATE)) + 1 AS INT) AS day_of_quarter,
  weekday(d) + 1                                          AS day_of_week,       -- 1 = Monday ... 7 = Sunday
  date_format(d, 'EEEE')                                  AS day_name,
  date_format(d, 'EEE')                                   AS day_short_name,
  -- Period boundaries
  CAST(date_trunc('WEEK', d) AS DATE)                     AS week_start_date,
  date_add(CAST(date_trunc('WEEK', d) AS DATE), 6)        AS week_end_date,
  trunc(d, 'MM')                                          AS month_start_date,
  last_day(d)                                             AS month_end_date,
  CAST(date_trunc('QUARTER', d) AS DATE)                  AS quarter_start_date,
  date_sub(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), 1) AS quarter_end_date,
  trunc(d, 'YEAR')                                        AS year_start_date,
  make_date(year(d), 12, 31)                              AS year_end_date,
  day(last_day(d))                                        AS days_in_month,
  -- Flags
  weekday(d) >= 5                                         AS is_weekend,
  weekday(d) < 5                                          AS is_weekday,
  d = trunc(d, 'MM')                                      AS is_month_start,
  d = last_day(d)                                         AS is_month_end,
  d = CAST(date_trunc('QUARTER', d) AS DATE)              AS is_quarter_start,
  d = date_sub(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), 1) AS is_quarter_end,
  (month(d) = 1 AND day(d) = 1)                           AS is_year_start,
  (month(d) = 12 AND day(d) = 31)                         AS is_year_end,
  ((year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0) AS is_leap_year
FROM dates
ORDER BY d;

In [ ]:
SELECT
  count(*)                  AS row_count,
  count(DISTINCT date_key)  AS distinct_date_keys,
  min(date)                 AS min_date,
  max(date)                 AS max_date,
  datediff(max(date), min(date)) + 1 AS expected_days
FROM adb_healthy.silver.calendar;

In [ ]:
select *
from adb_healthy.silver.calendar
limit 10
